# Change Data Feed (CDF) med Delta Sharing

Dette notebooken demonstrerer hvordan du bruker **Change Data Feed (CDF)** til å spore endringer i Delta-tabeller over tid.

In [ ]:
! pip install -r requirements.txt

In [ ]:
import os
import json
import delta_sharing
import pandas as pd
from google.cloud import storage
from datetime import datetime, timedelta

Vi henter config.share ved å kjøre "skyporten-deltashare"-repoet 

In [ ]:
# Spesifiser sti til din Delta Sharing config-fil
# Denne filen inneholder credentials og endpoint for din share som du henter ved å kjøre 
CONFIG_FILE = "share/config.share"  # Endre til din config-fil

In [ ]:
# Koble til Delta Sharing
sharing_client = delta_sharing.SharingClient(CONFIG_FILE)
tables = sharing_client.list_all_tables()

if not tables:
    raise Exception("❌ Ingen tabeller funnet i sharen")

not_valid_table_names = ["kode", "keys", "encrypted"]
valid_examples_tables = [
    table for table in tables
    if not any(substr in table.name for substr in not_valid_table_names)
]

# Velg første egnede tabell (eller første tabell hvis ingen egnede finnes)
table = valid_examples_tables[0] if len(valid_examples_tables) > 0 else tables[0]

# Bygg full tabell-URL for Delta Sharing
table_url = f"{CONFIG_FILE}#{table.share}.{table.schema}.{table.name}"

print(f"✓ Valgt tabell: {table.name}")
print(f"  Share: {table.share}")
print(f"  Schema: {table.schema}")
print(f"  Full URL: {table_url}")

##  Opprett Spark Session



In [ ]:
from pyspark.sql import SparkSession

def ensure_spark():
    global spark
    # Gjenbruk hvis 'spark' finnes og lever
    if 'spark' in globals():
        try:
            _ = spark.version
            print("Gjenbruker eksisterende SparkSession")
            return spark
        except Exception:
            pass  # faller gjennom og oppretter på nytt

    spark = (
        SparkSession.builder
        .appName("DeltaSharingCDF")
        # .master("local[*]")  # bruk bare lokalt; ikke i Databricks
        .config("spark.jars.packages", "io.delta:delta-sharing-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )
    return spark

spark = ensure_spark()


# Hent endringer via CDF basert på tidspunkt

1. Beregner et starttidspunkt basert på antall timer tilbake i tid.  
2. Leser endringer fra tabellen siden dette tidspunktet.  
3. Hvis det finnes endringer, grupperes de etter endringstype (`insert`, `update`, `delete`).  
4. Viser en oppsummering av antall endringer per type.  
5. Viser inntil 10 eksempler for hver endringstype.


In [ ]:
from pyspark.sql import SparkSession, functions as F
lookback_hours = 24
start_ts = (datetime.utcnow() - timedelta(hours=lookback_hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

cdf = (
    spark.read.format("deltaSharing")
    .option("responseFormat", "delta")
    .option("readChangeFeed", "true")
    .option("startingTimestamp", start_ts)
    .load(table_url)
)

if not cdf.rdd.isEmpty():
    change_summary = cdf.groupBy("_change_type").count().orderBy("_change_type")
    change_summary.show(truncate=False)

    for row in change_summary.collect():
        change_type = row["_change_type"]
        cdf.filter(F.col("_change_type") == change_type).show(10, truncate=False)
